In [1]:
import os
from dotenv import load_dotenv

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from openai import AsyncOpenAI, OpenAI

from openagv.core import SqliteAssetBin
from openagv.modules.audio import DeepgramAnalyzer

# Load environment variables from .env
load_dotenv()

async_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

chat_completion_service = OpenAIChatCompletion(
    service_id="my-service-id",
    ai_model_id="openai/gpt-4o-mini",
    async_client=async_client
)



In [2]:
# Goal is to be able to analyze songs, and analyze images, then render a 10 seconds video of flower sorted by color with the proper background song.

from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.modules.vision import ORVisionAnalyzer

# Define the instruction
instruct = UserInstruction("Analyze all the assets that haven't been analyzed and print it to debug. Including videos, transcribe them")

# Setup AssetBin and add assets
ab = SqliteAssetBin("db1.db")
ab.add_wildcard("examples/assets/*.png")
ab.add_wildcard("examples/assets/*.jpg")

# Initialize modules
vision_analyzer = ORVisionAnalyzer(client=openai_client, model='mistralai/ministral-8b-2512')
transcriber = DeepgramAnalyzer(api_key=os.getenv("DEEPGRAM_API_KEY"))

# Setup Executor
ex = SKLoopExecutor(ab, instruct, chat_completion=chat_completion_service, uses=[vision_analyzer, transcriber], debug=True)

# Execute
await ex.start()


[INFO] Starting real execution with instruction: 'Analyze all the assets that haven't been analyzed and print it to debug. Including videos, transcribe them'
[DEBUG] System Prompt: You are a helpful AI assistant capable of analyzing and manipulating media assets.
[DEBUG] Loaded Plugins: ['AssetBin', 'ORVisionAnalyzer', 'DeepgramAnalyzer']
[INFO] Advancing to step: AssetBin.list_possible_unanalyzed
[DEBUG] Invoking AssetBin.list_possible_unanalyzed with args: {}
[DEBUG] Result from AssetBin.list_possible_unanalyzed: No pending analyses found.
[INFO] Final Agent Response: There are no unanalyzed assets available at the moment.
[INFO] Execution finished.


In [3]:
ex.asset_bin.json_dump('dump.json')